# Analyze handscan validation inference

Metrics, threshold optimization, and ROC curves for DDIM→DDIM outputs on real SBND handscan labels:

- **healthy** = note `clean plane 1`
- **unhealthy** = note `streaks on plane 1`

Mirrors the scoring / ROC workflow in `AnalyzeDDIM2DDIM_Outputs.ipynb` and the comparison / summary style of `CompareROCCurves.ipynb`.

**Prerequisite:** pickles under `handscan_validation/inference_T100/{healthy,unhealthy}/` from `InferHandscanValidation.ipynb` / `parallel_handscan_inference.py`.

**Live ROC:** run the **Live ROC watcher** section below while workers are still writing pickles — it refreshes AUC / plots as files appear (`handscan_validation/roc_curves/live_*`).

**Kernel:** `env`.

In [1]:
from __future__ import annotations

import datetime
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc, roc_curve
from tqdm.auto import tqdm

stylefile = "presentation.mplstyle"
if Path(stylefile).exists():
    plt.style.use(stylefile)

# ── Configure ────────────────────────────────────────────────────────────────
RUN_HEALTHY = Path("handscan_validation/inference_T100/healthy")
RUN_UNHEALTHY = Path("handscan_validation/inference_T100/unhealthy")
T = 100
MODE = "ddim2ddim"

PROJECTION_AXIS = 0  # mean over ticks → one value per wire row
SCORE = "rms"        # "rms" | "proj_max_max" | "file_max_rms"
# file_max_rms: one score per pickle = max patch RMS (event-level ROC)

PATCH_ORDINAL = 0
ROC_SAVE_DIR = Path("handscan_validation/roc_curves")
ROC_LABEL = "SBND_handscan_streaks_plane1"
FIG_DIR = Path("handscan_validation/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 110
print("Healthy  ", RUN_HEALTHY.resolve())
print("Unhealthy", RUN_UNHEALTHY.resolve())

Healthy   /exp/sbnd/app/users/munjung/anomaly-detection/handscan_validation/inference_T100/healthy
Unhealthy /exp/sbnd/app/users/munjung/anomaly-detection/handscan_validation/inference_T100/unhealthy


In [5]:
def find_bundles(root: Path, T: int, mode: str = "ddim2ddim") -> list[Path]:
    return sorted(root.glob(f"*_T{T}_{mode}.pkl"))


def split_bundle(raw: dict) -> tuple[dict[int, dict], dict]:
    trace_keys = [k for k in raw if isinstance(k, str) and k.startswith("__") and k.endswith("_source__")]
    meta = raw.get(trace_keys[0], {}) if trace_keys else {}
    per_patch = {k: v for k, v in raw.items() if isinstance(k, int)}
    return per_patch, meta


def infer_keys(sample_patch: dict) -> tuple[str, str]:
    reco = [
        k for k in sample_patch
        if isinstance(k, str) and "-T" in k and not k.startswith("saliency") and k != "original"
    ]
    sal = [k for k in sample_patch if isinstance(k, str) and k.startswith("saliency-T")]
    if len(reco) != 1 or len(sal) != 1:
        raise ValueError(f"Could not infer keys (reco={reco}, saliency={sal})")
    return reco[0], sal[0]


def load_bundle(path: Path) -> tuple[dict[int, dict], dict, str, str]:
    with path.open("rb") as fh:
        raw = pickle.load(fh)
    per_patch, trace = split_bundle(raw)
    if not per_patch:
        raise ValueError(f"No patch entries in {path}")
    ddim_k, sal_k = infer_keys(per_patch[next(iter(per_patch))])
    return per_patch, trace, ddim_k, sal_k


def arrays_for_patch(bundle_path: Path, ordinal: int = 0):
    per_patch, trace, ddim_k, sal_k = load_bundle(bundle_path)
    keys_sorted = sorted(per_patch.keys())
    key = keys_sorted[ordinal % len(keys_sorted)]
    p = per_patch[key]
    return (
        np.squeeze(p["original"]),
        np.squeeze(p[ddim_k]),
        np.squeeze(p[sal_k]),
        trace,
        key,
    )


def projected_profile(abs_sal: np.ndarray, axis: int) -> np.ndarray:
    return np.mean(abs_sal, axis=axis)


healthy_pkls = find_bundles(RUN_HEALTHY, T, MODE)
unhealthy_pkls = find_bundles(RUN_UNHEALTHY, T, MODE)
print(f"Bundles: healthy={len(healthy_pkls)} unhealthy={len(unhealthy_pkls)}")

Bundles: healthy=2 unhealthy=3


## Live ROC watcher

Recompute the ROC as pickles appear under `inference_T100/{healthy,unhealthy}/` while the parallel workers run.

- Uses **file-level** score = max patch RMS (same as `live_roc_handscan.py`).
- Writes `handscan_validation/roc_curves/live_*.{json,png}` on each update.
- Set `LIVE_AUTO = True` to poll automatically; stop with the **Stop** button or Interrupt Kernel.

In [3]:
import time
from IPython.display import clear_output, display
import ipywidgets as widgets

# ── Live watcher config ───────────────────────────────────────────────────────
LIVE_POLL_SEC = 20
LIVE_AUTO = True          # False → only refresh when you click the button
LIVE_FILE_LEVEL = True    # True: max patch RMS per file (recommended while accumulating)
LIVE_MAX_ITERS = None     # e.g. 100 to auto-stop; None = run until Stop / interrupt

ROC_SAVE_DIR.mkdir(parents=True, exist_ok=True)

try:
    from live_roc_handscan import update_once as _live_update_once
except ImportError:
    _live_update_once = None


def _live_snapshot_key() -> tuple:
    h = tuple(p.name for p in find_bundles(RUN_HEALTHY, T, MODE))
    u = tuple(p.name for p in find_bundles(RUN_UNHEALTHY, T, MODE))
    return h, u


def live_roc_refresh(force: bool = False, _state: dict | None = None) -> dict:
    """Recompute live ROC if new pickles appeared (or force=True)."""
    state = _state if _state is not None else {}
    key = _live_snapshot_key()
    if not force and key == state.get("last_key"):
        return state.get("last_status") or {}

    if _live_update_once is not None:
        status = _live_update_once(
            RUN_HEALTHY, RUN_UNHEALTHY, ROC_SAVE_DIR, T, LIVE_FILE_LEVEL
        )
    else:
        # Inline fallback if the helper module is unavailable
        status = {"error": "live_roc_handscan.update_once not importable"}
        return status

    state["last_key"] = key
    state["last_status"] = status

    clear_output(wait=True)
    n_h = status.get("n_healthy_files", 0)
    n_u = status.get("n_unhealthy_files", 0)
    auc_val = status.get("auc")
    best = status.get("best_youden") or {}
    print(
        f"[{status.get('ts', '?')}] files healthy={n_h} unhealthy={n_u}  "
        f"scores h={status.get('n_healthy', 0)} u={status.get('n_unhealthy', 0)}  "
        f"AUC={auc_val if auc_val is not None else 'n/a'}"
    )
    if best.get("threshold") is not None:
        print(
            f"  Youden θ={best['threshold']:.6g}  "
            f"TPR={best.get('tpr')}  FPR={best.get('fpr')}  purity={best.get('purity')}"
        )

    # Show orchestrator status if present
    orch = Path("handscan_validation/run_state/orchestrator_status.json")
    if orch.is_file():
        try:
            os_ = json.loads(orch.read_text())
            print(
                f"  orchestrator: workers={os_.get('workers')}/{os_.get('max_workers')}  "
                f"queue={os_.get('queue')}  rss={os_.get('worker_rss_gb')} GiB  "
                f"done={os_.get('finished_ok')}  failed={len(os_.get('failed') or [])}"
            )
        except Exception as e:
            print(f"  (orchestrator status unreadable: {e})")

    roc_png = ROC_SAVE_DIR / "live_roc.png"
    scores_png = ROC_SAVE_DIR / "live_scores.png"
    if roc_png.is_file() and scores_png.is_file():
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(plt.imread(roc_png))
        axes[0].set_title("Live ROC")
        axes[0].axis("off")
        axes[1].imshow(plt.imread(scores_png))
        axes[1].set_title("Live score overlap")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
    elif n_h == 0 or n_u == 0:
        print("Waiting for at least one pickle in each class…")
    return status


_live_state: dict = {}
_live_stop = {"flag": False}

btn_refresh = widgets.Button(description="Refresh now", button_style="primary", icon="refresh")
btn_stop = widgets.Button(description="Stop auto-poll", button_style="danger", icon="stop")
live_status_html = widgets.HTML(value="<i>Live watcher idle</i>")


def _on_refresh(_):
    live_roc_refresh(force=True, _state=_live_state)
    live_status_html.value = f"<b>Last manual refresh</b> @ {_live_state.get('last_status', {}).get('ts', '?')}"


def _on_stop(_):
    _live_stop["flag"] = True
    live_status_html.value = "<b>Stop requested</b> — auto-poll will exit after this iteration."


btn_refresh.on_click(_on_refresh)
btn_stop.on_click(_on_stop)
display(widgets.HBox([btn_refresh, btn_stop]), live_status_html)

# Initial paint
live_roc_refresh(force=True, _state=_live_state)

if LIVE_AUTO:
    print(f"Auto-polling every {LIVE_POLL_SEC}s (Stop button or Interrupt Kernel to end)…")
    it = 0
    try:
        while not _live_stop["flag"]:
            time.sleep(LIVE_POLL_SEC)
            if _live_stop["flag"]:
                break
            live_roc_refresh(force=False, _state=_live_state)
            it += 1
            if LIVE_MAX_ITERS is not None and it >= LIVE_MAX_ITERS:
                print(f"Reached LIVE_MAX_ITERS={LIVE_MAX_ITERS}")
                break
            # Exit early if orchestrator finished and both classes complete
            orch = Path("handscan_validation/run_state/orchestrator_status.json")
            if orch.is_file():
                try:
                    os_ = json.loads(orch.read_text())
                    if os_.get("done") and os_.get("queue", 1) == 0 and os_.get("workers", 1) == 0:
                        live_roc_refresh(force=True, _state=_live_state)
                        print("Orchestrator done — stopping live watcher.")
                        break
                except Exception:
                    pass
    except KeyboardInterrupt:
        print("Live watcher interrupted.")
    live_status_html.value = "<b>Live watcher stopped</b>"
else:
    print("LIVE_AUTO=False — click Refresh now to update.")

[2026-09-02T15:38:36] files healthy=2 unhealthy=3  scores h=2 u=3  AUC=0.6666666666666666
  Youden θ=0.00933045  TPR=0.6666666666666666  FPR=0.0  purity=1.0
  orchestrator: workers=11/11  queue=31  rss=12.716 GiB  done=0  failed=0


/tmp/ipykernel_987852/893272006.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Live watcher interrupted.


## Example |saliency| projection + threshold illustration

In [9]:
if unhealthy_pkls:
    bp = Path(unhealthy_pkls[0])
    _, _, sal_example, trace_ex, pk = arrays_for_patch(bp, PATCH_ORDINAL)
    abs_s = np.abs(sal_example.astype(np.float64))
    proj = projected_profile(abs_s, PROJECTION_AXIS)
    thresh_plot = float(np.percentile(proj, 95))
    flagged = proj >= thresh_plot

    fig, ax = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw={"height_ratios": [1.2, 1]})
    im = ax[0].imshow(abs_s, aspect="auto", cmap="inferno", origin="lower")
    plt.colorbar(im, ax=ax[0], fraction=0.035, label="|saliency|")
    ax[0].set_title(f"Example |saliency| patch key={pk} — {bp.name}")

    xc = np.arange(proj.size)
    ax[1].plot(xc, proj, lw=1, label="projection: mean |saliency|")
    ax[1].axhline(thresh_plot, color="orange", ls="--", lw=2, label=f"θ = P95 = {thresh_plot:.4g}")
    ax[1].scatter(xc[flagged], proj[flagged], c="cyan", s=10, alpha=0.7, label="flagged")
    ax[1].set_xlabel("row index" if PROJECTION_AXIS == 0 else "column index")
    ax[1].set_ylabel("projected mean |saliency|")
    ax[1].legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "example_saliency_projection.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("No unhealthy pickles found yet.")

/tmp/ipykernel_987852/1646373355.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Score distributions, ROC, efficiency × purity, optimal threshold

In [10]:
def bundle_patch_scores(bundle_path: Path, axis: int, score_mode: str) -> list[float]:
    per_patch, _, _, sal_k = load_bundle(bundle_path)
    patch_scores: list[float] = []
    for j in sorted(per_patch.keys()):
        sal = np.squeeze(per_patch[j][sal_k]).astype(np.float64)
        abs_s = np.abs(sal)
        if score_mode in ("rms", "file_max_rms"):
            patch_scores.append(float(np.sqrt(np.mean(abs_s**2))))
        elif score_mode == "proj_max_max":
            patch_scores.append(float(np.max(projected_profile(abs_s, axis))))
        else:
            raise ValueError(score_mode)
    return patch_scores


def collect_scores(pkls: list[Path], score_mode: str) -> np.ndarray:
    scores: list[float] = []
    for bp in tqdm(pkls, desc=f"score {score_mode}"):
        try:
            ps = bundle_patch_scores(bp, PROJECTION_AXIS, score_mode)
            if score_mode == "file_max_rms":
                scores.append(float(np.max(ps)) if ps else np.nan)
            else:
                scores.extend(ps)
        except Exception as e:
            print("skip", bp, e)
    return np.asarray([s for s in scores if np.isfinite(s)], dtype=float)


healthy_scores = collect_scores(healthy_pkls, SCORE)
unhealthy_scores = collect_scores(unhealthy_pkls, SCORE)

y = np.concatenate([np.zeros(len(healthy_scores)), np.ones(len(unhealthy_scores))])
scores_all = np.concatenate([healthy_scores, unhealthy_scores])

print("n healthy", len(healthy_scores), "n unhealthy", len(unhealthy_scores))

fpr = tpr = thresholds_roc = None
auc_val = float("nan")
best = {}

if scores_all.size and np.unique(y).size >= 2:
    fpr, tpr, thresholds_roc = roc_curve(y, scores_all)
    auc_val = float(auc(fpr, tpr))

    # Sweep unique score cuts (descending) for efficiency / purity / Youden J
    cuts = np.sort(np.unique(scores_all))[::-1]
    rows = []
    for th in cuts:
        pred = scores_all >= th
        tp = int(np.sum(pred & (y == 1)))
        fp = int(np.sum(pred & (y == 0)))
        tn = int(np.sum(~pred & (y == 0)))
        fn = int(np.sum(~pred & (y == 1)))
        eff = tp / (tp + fn) if (tp + fn) else np.nan
        pur = tp / (tp + fp) if (tp + fp) else np.nan
        fpr_th = fp / (fp + tn) if (fp + tn) else np.nan
        j = (eff - fpr_th) if np.isfinite(eff) and np.isfinite(fpr_th) else -np.inf
        f1 = (2 * pur * eff / (pur + eff)) if (pur + eff) and np.isfinite(pur) and np.isfinite(eff) else np.nan
        rows.append({"threshold": float(th), "eff": eff, "pur": pur, "fpr": fpr_th, "youden_j": j, "f1": f1,
                     "tp": tp, "fp": fp, "tn": tn, "fn": fn})

    best_j = max(rows, key=lambda r: r["youden_j"])
    best_f1 = max(rows, key=lambda r: (r["f1"] if np.isfinite(r["f1"]) else -1))
    best = {"youden": best_j, "f1": best_f1, "auc": auc_val}

    print(f"AUC = {auc_val:.4f}")
    print(f"Best Youden J: θ={best_j['threshold']:.6g}  eff={best_j['eff']:.3f}  "
          f"pur={best_j['pur']:.3f}  FPR={best_j['fpr']:.3f}  J={best_j['youden_j']:.3f}")
    print(f"Best F1:       θ={best_f1['threshold']:.6g}  eff={best_f1['eff']:.3f}  "
          f"pur={best_f1['pur']:.3f}  F1={best_f1['f1']:.3f}")

    # ROC + efficiency×purity
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.8))
    ax[0].plot(fpr, tpr, lw=2, label=f"ROC AUC={auc_val:.4f}")
    ax[0].plot([0, 1], [0, 1], ls="--", color="grey")
    ax[0].scatter([best_j["fpr"]], [best_j["eff"]], c="crimson", zorder=5,
                  label=f"Youden θ={best_j['threshold']:.4g}")
    ax[0].set_xlabel("False-positive rate")
    ax[0].set_ylabel("True-positive rate")
    ax[0].set_title("Handscan validation ROC")
    ax[0].legend(fontsize=9)
    ax[0].set_aspect("equal", adjustable="box")

    effs = np.array([r["eff"] for r in rows])
    purs = np.array([r["pur"] for r in rows])
    sel = ~(np.isnan(effs) | np.isnan(purs))
    ax[1].plot(effs[sel], purs[sel], lw=2)
    ax[1].scatter([best_j["eff"]], [best_j["pur"]], c="crimson", zorder=5, label="Youden")
    ax[1].scatter([best_f1["eff"]], [best_f1["pur"]], c="navy", zorder=5, label="best F1")
    ax[1].set_xlim(0, 1.03)
    ax[1].set_ylim(0, 1.03)
    ax[1].set_xlabel("Efficiency (unhealthy flagged)")
    ax[1].set_ylabel("Purity TP/(TP+FP)")
    ax[1].set_title("Efficiency × purity")
    ax[1].grid(True, alpha=0.3)
    ax[1].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"roc_effpur_{SCORE}_T{T}.png", bbox_inches="tight", dpi=150)
    plt.show()

    # Score overlap
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = 30
    ax.hist(healthy_scores, bins=bins, alpha=0.6, label="Healthy (clean plane 1)", color="teal")
    ax.hist(unhealthy_scores, bins=bins, alpha=0.55, label="Unhealthy (streaks plane 1)", color="coral")
    ax.axvline(best_j["threshold"], color="crimson", ls="--", label=f"Youden θ={best_j['threshold']:.4g}")
    ax.set_xlabel(f"Score ({SCORE})")
    ax.set_ylabel("Count")
    ax.set_title(f"Score overlap — T={T}, {MODE}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"score_overlap_{SCORE}_T{T}.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("Need nonempty scores for both classes. Run inference first.")

score rms:   0%|          | 0/2 [00:00<?, ?it/s]

score rms:   0%|          | 0/3 [00:00<?, ?it/s]

n healthy 12 n unhealthy 18
AUC = 0.6065
Best Youden J: θ=0.00635457  eff=0.333  pur=0.857  FPR=0.083  J=0.250
Best F1:       θ=0.00492723  eff=0.889  pur=0.667  F1=0.762


/tmp/ipykernel_987852/1181418362.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_987852/1181418362.py:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sample inspection — original / noised / reconstructed / difference / RMS

For a chosen processed pickle + patch, show the DDIM→DDIM pipeline side-by-side.

- **original** — input patch  
- **noised** — DDIM forward state at `T` (only if saved in the pickle; current inference stores original / reco / saliency only)  
- **reconstructed** — DDIM inverse estimate  
- **difference** — `reco − original` (signed saliency)  
- **RMS map** — `|saliency|` (the field whose spatial RMS is the anomaly score)

Set `INSPECT_PKL` / `INSPECT_PATCH` below, or leave `None` to auto-pick an unhealthy example.

In [11]:
def load_patch_images(bundle_path: Path, ordinal: int = 0):
    """Return (orig, noised_or_None, reco, sal, rms_map, rms_score, trace, patch_key)."""
    per_patch, trace, ddim_k, sal_k = load_bundle(bundle_path)
    keys_sorted = sorted(per_patch.keys())
    key = keys_sorted[ordinal % len(keys_sorted)]
    p = per_patch[key]
    orig = np.squeeze(p["original"]).astype(np.float64)
    reco = np.squeeze(p[ddim_k]).astype(np.float64)
    sal = np.squeeze(p[sal_k]).astype(np.float64)
    # Optional intermediates some savers may store
    noised = None
    for nk in ("noised", "ddim_noised", f"noised-T{T}", "diffused"):
        if nk in p:
            noised = np.squeeze(p[nk]).astype(np.float64)
            break
    rms_map = np.abs(sal)
    rms_score = float(np.sqrt(np.mean(rms_map**2)))
    return orig, noised, reco, sal, rms_map, rms_score, trace, key


def plot_pipeline_panels(
    orig, noised, reco, sal, rms_map, *, title: str, rms_score: float, save_path: Path | None = None
):
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
    panels = [
        (axes[0], orig, "Original", "bwr", -0.5, 0.5),
        (axes[1], noised, "Noised (DDIM @ T)", "bwr", -0.5, 0.5),
        (axes[2], reco, "Reconstructed", "bwr", -0.5, 0.5),
        (axes[3], sal, "Difference (reco − orig)", "bwr", -0.15, 0.15),
        (axes[4], rms_map, f"|saliency|  RMS={rms_score:.4g}", "inferno", None, None),
    ]
    for ax, img, lab, cmap, v0, v1 in panels:
        if img is None:
            ax.set_facecolor("#f0f0f0")
            ax.text(0.5, 0.5, "not stored\nin pickle", ha="center", va="center", fontsize=11, transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(lab)
            continue
        kw = dict(aspect="auto", origin="lower", cmap=cmap)
        if v0 is not None:
            kw["vmin"], kw["vmax"] = v0, v1
        im = ax.imshow(np.squeeze(img), **kw)
        ax.set_title(lab, fontsize=10)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()


# ── pick a sample ─────────────────────────────────────────────────────────────
healthy_pkls = find_bundles(RUN_HEALTHY, T, MODE)
unhealthy_pkls = find_bundles(RUN_UNHEALTHY, T, MODE)

INSPECT_PKL: Path | None = None   # e.g. unhealthy_pkls[0]
INSPECT_PATCH = PATCH_ORDINAL
INSPECT_LABEL = "unhealthy"       # used only when INSPECT_PKL is None

if INSPECT_PKL is None:
    pool = unhealthy_pkls if INSPECT_LABEL == "unhealthy" else healthy_pkls
    if not pool:
        pool = healthy_pkls or unhealthy_pkls
    INSPECT_PKL = pool[0] if pool else None

if INSPECT_PKL is None:
    print("No pickles available for inspection yet.")
else:
    orig, noised, reco, sal, rms_map, rms_score, trace, pk = load_patch_images(
        Path(INSPECT_PKL), INSPECT_PATCH
    )
    fn = trace.get("input_filename", Path(INSPECT_PKL).name)
    label = trace.get("label", "?")
    title = f"{label} | {fn} | patch={pk} | T={T}"
    if noised is None:
        print("Note: noised intermediate not in pickle (inference saves original / reco / saliency only).")
    plot_pipeline_panels(
        orig, noised, reco, sal, rms_map,
        title=title,
        rms_score=rms_score,
        save_path=FIG_DIR / f"pipeline_{Path(INSPECT_PKL).stem}_p{pk}.png",
    )

Note: noised intermediate not in pickle (inference saves original / reco / saliency only).


/tmp/ipykernel_987852/1619380844.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Confusion examples — TP / FP / TN / FN

Using the **Youden** threshold from the ROC cell above (falls back to median score if ROC has not been run), classify each scored unit and plot a few examples per class.

- **TP** — unhealthy, score ≥ θ (correctly flagged)  
- **FP** — healthy, score ≥ θ (false alarm)  
- **TN** — healthy, score < θ (correctly clean)  
- **FN** — unhealthy, score < θ (missed streaks)

Each example shows original + |saliency| (RMS map). Set `N_CONFUSION_EXAMPLES` for how many rows per class.

In [12]:
def iter_scored_units(pkls: list[Path], y_true: int, score_mode: str):
    """Yield dicts with path, patch_key, score, y_true, arrays for plotting."""
    for bp in pkls:
        try:
            per_patch, trace, ddim_k, sal_k = load_bundle(bp)
        except Exception as e:
            print("skip", bp, e)
            continue
        patch_items = []
        for j in sorted(per_patch.keys()):
            sal = np.squeeze(per_patch[j][sal_k]).astype(np.float64)
            abs_s = np.abs(sal)
            if score_mode in ("rms", "file_max_rms"):
                sc = float(np.sqrt(np.mean(abs_s**2)))
            elif score_mode == "proj_max_max":
                sc = float(np.max(projected_profile(abs_s, PROJECTION_AXIS)))
            else:
                raise ValueError(score_mode)
            patch_items.append((j, sc, per_patch[j], ddim_k, sal_k))

        if score_mode == "file_max_rms":
            if not patch_items:
                continue
            j, sc, p, ddim_k, sal_k = max(patch_items, key=lambda t: t[1])
            yield {
                "path": bp,
                "patch_key": j,
                "score": sc,
                "y_true": y_true,
                "original": np.squeeze(p["original"]),
                "reco": np.squeeze(p[ddim_k]),
                "sal": np.squeeze(p[sal_k]),
                "trace": trace,
            }
        else:
            for j, sc, p, ddim_k, sal_k in patch_items:
                yield {
                    "path": bp,
                    "patch_key": j,
                    "score": sc,
                    "y_true": y_true,
                    "original": np.squeeze(p["original"]),
                    "reco": np.squeeze(p[ddim_k]),
                    "sal": np.squeeze(p[sal_k]),
                    "trace": trace,
                }


N_CONFUSION_EXAMPLES = 3

# Refresh file lists in case inference advanced since earlier cells
healthy_pkls = find_bundles(RUN_HEALTHY, T, MODE)
unhealthy_pkls = find_bundles(RUN_UNHEALTHY, T, MODE)

records = list(iter_scored_units(healthy_pkls, 0, SCORE)) + list(
    iter_scored_units(unhealthy_pkls, 1, SCORE)
)

if not records:
    print("No scored records — run inference first.")
elif "best" in dir() and best.get("youden") and best["youden"].get("threshold") is not None:
    theta = float(best["youden"]["threshold"])
    theta_src = "Youden"
else:
    theta = float(np.median([r["score"] for r in records]))
    theta_src = "median fallback"

buckets: dict[str, list] = {"TP": [], "FP": [], "TN": [], "FN": []}
for r in records:
    pred = r["score"] >= theta
    if r["y_true"] == 1 and pred:
        buckets["TP"].append(r)
    elif r["y_true"] == 0 and pred:
        buckets["FP"].append(r)
    elif r["y_true"] == 0 and not pred:
        buckets["TN"].append(r)
    else:
        buckets["FN"].append(r)

# Prefer extreme scores within each bucket (most confident mistakes / hits)
for name, lst in buckets.items():
    if name in ("TP", "FP"):
        lst.sort(key=lambda r: r["score"], reverse=True)
    else:
        lst.sort(key=lambda r: r["score"])

print(f"Threshold θ={theta:.6g} ({theta_src}), score={SCORE}")
for name in ("TP", "FP", "TN", "FN"):
    print(f"  {name}: {len(buckets[name])}")


def _show_confusion_row(examples: list, class_name: str):
    n = len(examples)
    if n == 0:
        print(f"{class_name}: none")
        return
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.2 * n), squeeze=False)
    for i, r in enumerate(examples):
        orig = np.squeeze(r["original"])
        sal = np.squeeze(r["sal"]).astype(np.float64)
        abs_s = np.abs(sal)
        rms = float(np.sqrt(np.mean(abs_s**2)))
        fn = (r.get("trace") or {}).get("input_filename", r["path"].name)
        im0 = axes[i, 0].imshow(orig, aspect="auto", origin="lower", cmap="bwr", vmin=-0.5, vmax=0.5)
        axes[i, 0].set_title("Original", fontsize=9)
        im1 = axes[i, 1].imshow(np.squeeze(r["reco"]), aspect="auto", origin="lower", cmap="bwr", vmin=-0.5, vmax=0.5)
        axes[i, 1].set_title("Reconstructed", fontsize=9)
        im2 = axes[i, 2].imshow(abs_s, aspect="auto", origin="lower", cmap="inferno")
        axes[i, 2].set_title(f"|saliency|  score={r['score']:.4g}", fontsize=9)
        for ax in axes[i]:
            ax.set_xticks([])
            ax.set_yticks([])
        fig.colorbar(im0, ax=axes[i, 0], fraction=0.046, pad=0.04)
        fig.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)
        fig.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)
        axes[i, 0].set_ylabel(f"p{r['patch_key']}\n{fn[:40]}", fontsize=8)
    fig.suptitle(f"{class_name}  (θ={theta:.4g}, n={n})", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"confusion_{class_name}_{SCORE}_T{T}.png", bbox_inches="tight", dpi=150)
    plt.show()


for name in ("TP", "FP", "TN", "FN"):
    _show_confusion_row(buckets[name][:N_CONFUSION_EXAMPLES], name)

Threshold θ=0.00635457 (Youden), score=rms
  TP: 7
  FP: 2
  TN: 16
  FN: 17


/tmp/ipykernel_987852/1212872022.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save ROC JSON (CompareROCCurves-compatible)

In [13]:
if ROC_SAVE_DIR is not None and fpr is not None:
    ROC_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    out_path = ROC_SAVE_DIR / f"{ROC_LABEL}_T{T}_{SCORE}_{MODE}.json"
    payload = {
        "label": ROC_LABEL,
        "T": T,
        "score_mode": SCORE,
        "projection_axis": PROJECTION_AXIS,
        "nominal_dir": str(RUN_HEALTHY),
        "defect_dir": str(RUN_UNHEALTHY),
        "n_nominal_patches": int(len(healthy_scores)),
        "n_defect_patches": int(len(unhealthy_scores)),
        "auc": float(auc_val),
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
        "best_youden": best.get("youden"),
        "best_f1": best.get("f1"),
        "timestamp": timestamp,
        "healthy_note": "clean plane 1",
        "unhealthy_note": "streaks on plane 1",
        "wire_crop": [3600, 4000],
    }
    with out_path.open("w") as fh:
        json.dump(payload, fh, indent=2)
    print(f"Saved ROC → {out_path}  (AUC={auc_val:.4f})")
else:
    print("ROC saving skipped.")

Saved ROC → handscan_validation/roc_curves/SBND_handscan_streaks_plane1_T100_rms_ddim2ddim.json  (AUC=0.6065)


## Optional: overlay multiple saved ROC curves

Same idea as `CompareROCCurves.ipynb` — load JSONs from `ROC_SAVE_DIR` and plot together.

In [14]:
from scipy.ndimage import gaussian_filter1d


def smooth_roc(fpr_arr: np.ndarray, tpr_arr: np.ndarray, sigma: float):
    if sigma <= 0:
        return fpr_arr, tpr_arr
    tpr_s = np.clip(gaussian_filter1d(tpr_arr, sigma=sigma), 0.0, 1.0)
    tpr_s[0], tpr_s[-1] = tpr_arr[0], tpr_arr[-1]
    return fpr_arr, tpr_s


roc_files = sorted(ROC_SAVE_DIR.glob("*.json")) if ROC_SAVE_DIR.exists() else []
records = []
for path in roc_files:
    with path.open() as fh:
        rec = json.load(fh)
    records.append(rec)
    print(f"  {path.name}  AUC={rec.get('auc', float('nan')):.4f}")

if records:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1, label="Random")
    for rec in records:
        fpr_r = np.asarray(rec["fpr"])
        tpr_r = np.asarray(rec["tpr"])
        fpr_r, tpr_r = smooth_roc(fpr_r, tpr_r, 1)
        ax.plot(fpr_r, tpr_r, lw=2, label=f"{rec.get('label','?')} (AUC={rec.get('auc', float('nan')):.2f})")
    ax.set_xlabel("False-positive rate")
    ax.set_ylabel("True-positive rate")
    ax.set_title("Handscan validation — streak detection")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "roc_overlay.png", bbox_inches="tight", dpi=150)
    plt.show()

    header = f"{'Label':<40} {'T':>5} {'Score':<14} {'n_h':>7} {'n_u':>7} {'AUC':>8}"
    print(header)
    print("-" * len(header))
    for rec in sorted(records, key=lambda r: r.get("auc", 0), reverse=True):
        print(
            f"{rec.get('label', '?'):<40}"
            f" {str(rec.get('T', '?')):>5}"
            f" {rec.get('score_mode', '?'):<14}"
            f" {rec.get('n_nominal_patches', '?'):>7}"
            f" {rec.get('n_defect_patches', '?'):>7}"
            f" {rec.get('auc', float('nan')):>8.4f}"
        )
else:
    print("No ROC JSON files to overlay yet.")

  SBND_handscan_streaks_plane1_T100_rms_ddim2ddim.json  AUC=0.6065
  live_roc.json  AUC=0.5833
  live_status.json  AUC=0.5833


KeyError: 'fpr'